## Importing packages

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, TimeSeriesSplit
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    mean_absolute_error,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import TransformedTargetRegressor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.model_selection import KFold

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt
import joblib
from datetime import datetime

import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Data Loading & Preparation

In [ ]:
df = pd.read_csv("../data_model/db_additional_fe.csv")
df_test = pd.read_csv("../data_model/db_collected_fe.csv")

def sanitize_column_names(df):
    """Replace special JSON characters in column names with underscores."""
    import re
    sanitized = {}
    for col in df.columns:
        sanitized_col = re.sub(r'[\[\](){},\"\'\\\/\s]', '_', str(col))
        sanitized_col = re.sub(r'_+', '_', sanitized_col)
        sanitized_col = sanitized_col.strip('_')
        sanitized[col] = sanitized_col
    df = df.rename(columns=sanitized)
    return df

df = sanitize_column_names(df)
df_test = sanitize_column_names(df_test)

print(f"Data loaded: Train {df.shape}, Test {df_test.shape}")

Data loaded: Train (13, 50), Test (652220, 50)


## REGRESSION

In [17]:
X_train_raw = df.drop(columns=['volume_total'])
y_train_raw = df['volume_total']

X_test_raw = df_test.drop(columns=['volume_total'])
y_test_raw = df_test['volume_total']

print(f"Raw data - Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")
print(f"y_train range: [{y_train_raw.min():.0f}, {y_train_raw.max():.0f}]")
print(f"Non-zero values in y_train: {(y_train_raw > 0).sum()}")

X_train_filled = X_train_raw.fillna(0)
X_test_filled = X_test_raw.fillna(0)

common_cols = X_train_filled.columns.intersection(X_test_filled.columns)
print(f"Aligning columns: {len(X_train_filled.columns)} train, {len(X_test_filled.columns)} test → {len(common_cols)} common")
X_train_filled = X_train_filled[common_cols]
X_test_filled = X_test_filled[common_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

print("\n Selecting top 150 features from", X_train_scaled.shape[1])
selector = SelectKBest(f_regression, k=150)
X_train_selected = selector.fit_transform(X_train_scaled, y_train_raw)
X_test_selected = selector.transform(X_test_scaled)

print(f" Selected features: {X_train_selected.shape[1]}")

selected_indices = selector.get_support(indices=True)
selected_features = X_train_raw.columns[selected_indices].tolist()

print("\n Log-transforming target...")
y_train_log = np.log1p(y_train_raw)
y_test_log = np.log1p(y_test_raw)

print(f"Original y_train range: [{y_train_raw.min():.0f}, {y_train_raw.max():.0f}]")
print(f"Log-transformed y_train range: [{y_train_log.min():.2f}, {y_train_log.max():.2f}]")
print(f"Log-transform makes scaling uniform")

Raw data - Train: (13, 49), Test: (652220, 49)
y_train range: [6, 13430]
Non-zero values in y_train: 13
Aligning columns: 49 train, 49 test → 49 common


ValueError: could not convert string to float: '2022-12-13 23:36:45'

In [ ]:
RANDOM_STATE = 2026
N_JOBS = -1
N_TRIALS = 10  
CV_FOLDS = 2   

param_spaces = {
    "xgb": {
        "n_estimators": ("int", 300, 800),   
        "max_depth": ("int", 4, 8),         
        "learning_rate": ("float", 0.01, 0.1),
        "subsample": ("float", 0.7, 1.0),
        "colsample_bytree": ("float", 0.7, 1.0),
        "min_child_weight": ("int", 1, 5),   
        "gamma": ("float", 0.1, 1.5),          
        "reg_alpha": ("float", 0.1, 2.0),    
        "reg_lambda": ("float", 0.5, 5.0),       
    },
    
    "lgbm": {
        "n_estimators": ("int", 300, 800),
        "max_depth": ("int", 5, 10),   
        "learning_rate": ("float", 0.01, 0.1),
        "num_leaves": ("int", 20, 100),   
        "subsample": ("float", 0.7, 1.0),
        "colsample_bytree": ("float", 0.7, 1.0),
        "min_child_samples": ("int", 10, 50),
        "reg_alpha": ("float", 0.1, 2.0),
        "reg_lambda": ("float", 0.5, 5.0),
    },
    
    "rf": {
        "n_estimators": ("int", 300, 800),
        "max_depth": ("int", 5, 15),        
        "min_samples_split": ("int", 5, 20),
        "min_samples_leaf": ("int", 2, 10),
        "max_features": ("categorical", ["sqrt", "log2"]),
    },
}

model_constructors = {
    "xgb": lambda params: XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "lgbm": lambda params: LGBMRegressor(
        objective="regression",
        verbose=-1, 
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "rf": lambda params: RandomForestRegressor(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
}

print(f"\n Configuration ready:")
print(f"  - N_TRIALS: {N_TRIALS}")
print(f"  - CV_FOLDS: {CV_FOLDS}")
print(f"  - Selected features: {X_train_selected.shape[1]}")
print(f"  - Target transformed: log(1 + volume)")


 Configuration ready:
  - N_TRIALS: 10
  - CV_FOLDS: 2
  - Selected features: 150
  - Target transformed: log(1 + volume)


In [ ]:
def get_model_type(name):
    """Extract model type from name."""
    if "xgb" in name:
        return "xgb"
    elif "lgbm" in name:
        return "lgbm"
    elif "rf" in name:
        return "rf"
    return None

def create_regression_objective(model_name, X_train, y_train, cv):
    """Create Optuna objective for regression."""
    
    def objective(trial):
        params = {}
        for param_name, param_config in param_spaces[model_name].items():
            param_type = param_config[0]
            
            if param_type == "int":
                params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
            elif param_type == "float":
                params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2])
            elif param_type == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_config[1])
        
        model = model_constructors[model_name](params)
        
        scores = cross_val_score(
            model, X_train, y_train,
            cv=cv,
            scoring="neg_mean_squared_error",
            n_jobs=N_JOBS
        )
        
        rmse = np.sqrt(-scores.mean())
        return rmse
    
    return objective

print(" Objective functions ready")

 Objective functions ready


In [ ]:
cv_regression = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_regression = {}

for model_name in ["xgb", "lgbm", "rf"]:
    print(f"\n{'='*60}")
    print(f"Training: {model_name.upper()}")
    print(f"{'='*60}")
    
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{model_name}_regression"
    )
    
    objective = create_regression_objective(model_name, X_train_selected, y_train_log, cv_regression)
    
    print(f"Running {N_TRIALS} trials...")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    best_params = study.best_params
    
    # Train final model with best params
    model = model_constructors[model_name](best_params)
    model.fit(X_train_selected, y_train_log)
    
    # Predict on test
    y_pred_log = model.predict(X_test_selected)
    y_pred = np.expm1(y_pred_log)  # Transform back to original scale
    
    # Metrics on ORIGINAL scale
    rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred))
    mae = mean_absolute_error(y_test_raw, y_pred)
    r2 = 1 - (np.sum((y_test_raw - y_pred)**2) / np.sum((y_test_raw - y_test_raw.mean())**2))
    
    results_regression[model_name] = {
        "study": study,
        "model": model,
        "best_params": best_params,
        "cv_rmse": study.best_value,
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
    }
    
    print(f"  CV RMSE (log scale): {study.best_value:.4f}")
    print(f"  Test RMSE (original): {rmse:,.0f}")
    print(f"  Test MAE: {mae:,.0f}")
    print(f"  Test R²: {r2:.4f}")

[I 2026-01-27 21:10:36,175] A new study created in memory with name: xgb_regression



Training: XGB
Running 10 trials...


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-01-27 21:10:37,585] Trial 0 finished with value: 2.184869731450633 and parameters: {'n_estimators': 409, 'max_depth': 6, 'learning_rate': 0.09789719303442712, 'subsample': 0.7266697064944387, 'colsample_bytree': 0.8437869579040544, 'min_child_weight': 5, 'gamma': 0.3773975246264854, 'reg_alpha': 1.8322588455060218, 'reg_lambda': 3.0121255168963623}. Best is trial 0 with value: 2.184869731450633.
[I 2026-01-27 21:10:38,677] Trial 1 finished with value: 2.038722375553254 and parameters: {'n_estimators': 692, 'max_depth': 5, 'learning_rate': 0.04575708941376109, 'subsample': 0.7887901944071268, 'colsample_bytree': 0.96822653830435, 'min_child_weight': 1, 'gamma': 0.7230946998690196, 'reg_alpha': 1.1675983894845703, 'reg_lambda': 2.1367158361954792}. Best is trial 1 with value: 2.038722375553254.
[I 2026-01-27 21:10:39,786] Trial 2 finished with value: 2.0961190536781737 and parameters: {'n_estimators': 647, 'max_depth': 8, 'learning_rate': 0.012753376078799563, 'subsample': 0.7364

[I 2026-01-27 21:10:41,265] A new study created in memory with name: lgbm_regression


  CV RMSE (log scale): 2.0142
  Test RMSE (original): 1,562
  Test MAE: 301
  Test R²: 0.4604

Training: LGBM
Running 10 trials...


... (2 outputs left out)

[I 2026-01-27 21:10:44,310] A new study created in memory with name: rf_regression


[I 2026-01-27 21:10:44,230] Trial 9 finished with value: 2.0096523886136195 and parameters: {'n_estimators': 728, 'max_depth': 9, 'learning_rate': 0.0923974670981009, 'num_leaves': 42, 'subsample': 0.75040197720529, 'colsample_bytree': 0.7697348152745916, 'min_child_samples': 43, 'reg_alpha': 1.7200380182407462, 'reg_lambda': 0.9207109860315543}. Best is trial 0 with value: 2.0096523886136195.
  CV RMSE (log scale): 2.0097
  Test RMSE (original): 1,700
  Test MAE: 382
  Test R²: 0.3609

Training: RF
Running 10 trials...


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-01-27 21:10:44,795] Trial 0 finished with value: 2.0086757464225338 and parameters: {'n_estimators': 409, 'max_depth': 9, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 2.0086757464225338.
[I 2026-01-27 21:10:45,190] Trial 1 finished with value: 1.9949992469792819 and parameters: {'n_estimators': 399, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 1 with value: 1.9949992469792819.
[I 2026-01-27 21:10:45,440] Trial 2 finished with value: 1.9823312734308247 and parameters: {'n_estimators': 448, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 2 with value: 1.9823312734308247.
[I 2026-01-27 21:10:45,783] Trial 3 finished with value: 2.031820464665437 and parameters: {'n_estimators': 647, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 2 with value: 1.98233127343082

In [ ]:
print(f"\n{'='*70}")
print(f"{'='*20} REGRESSION SUMMARY {'='*20}")
print(f"{'='*70}\n")

print(f"{'Model':<12} {'CV RMSE':<15} {'Test RMSE':<15} {'MAE':<15} {'R²':<10}")
print("-" * 70)

for name, result in results_regression.items():
    cv = result['cv_rmse']
    test = result['test_rmse']
    mae = result['test_mae']
    r2 = result['test_r2']
    print(f"{name:<12} {cv:<15.2f} {test:<15,.0f} {mae:<15,.0f} {r2:<10.4f}")

best_reg_name = min(results_regression.items(), key=lambda x: x[1]['test_rmse'])[0]
print(f"\n Best model by Test RMSE: {best_reg_name.upper()} (RMSE: {results_regression[best_reg_name]['test_rmse']:,.0f})")


==================== REGRESSION SUMMARY ====================

Model        CV RMSE         Test RMSE       MAE             R²        
----------------------------------------------------------------------
xgb          2.01            1,562           301             0.4604    
lgbm         2.01            1,700           382             0.3609    
rf           1.98            2,112           546             0.0134    

 Best model by Test RMSE: XGB (RMSE: 1,562)


## CLASSIFICATION

In [ ]:
y_train_clf = (df['volume_total'] > 10_000).astype(int)
X_train_clf = df.drop(columns=['volume_total'])

y_test_clf = (df_test['volume_total'] > 10_000).astype(int)
X_test_clf = df_test.drop(columns=['volume_total'])

print(f"Classification data - Train: {X_train_clf.shape}, Test: {X_test_clf.shape}")
print(f"Train distribution: {(y_train_clf==0).sum()} negative ({(y_train_clf==0).sum()/len(y_train_clf):.1%}), "
      f"{(y_train_clf==1).sum()} positive ({(y_train_clf==1).sum()/len(y_train_clf):.1%})")
print(f"Test distribution: {(y_test_clf==0).sum()} negative ({(y_test_clf==0).sum()/len(y_test_clf):.1%}), "
      f"{(y_test_clf==1).sum()} positive ({(y_test_clf==1).sum()/len(y_test_clf):.1%})")

X_train_clf = X_train_clf.fillna(0)
X_test_clf = X_test_clf.fillna(0)

Classification data - Train: (51, 1942), Test: (51, 1942)
Train distribution: 50 negative (98.0%), 1 positive (2.0%)
Test distribution: 50 negative (98.0%), 1 positive (2.0%)


In [ ]:
pos_count = (y_train_clf == 1).sum()
neg_count = (y_train_clf == 0).sum()
total_count = len(y_train_clf)

current_pos_ratio = pos_count / total_count
current_neg_ratio = neg_count / total_count

target_pos_ratio = 0.05 #from test set
target_neg_ratio = 0.95 #from test set

weight_pos = target_pos_ratio / current_pos_ratio
weight_neg = target_neg_ratio / current_neg_ratio

sample_weights = np.where(y_train_clf == 1, weight_pos, weight_neg)

print(f"Train distribution: {neg_count} negative ({current_neg_ratio:.1%}), {pos_count} positive ({current_pos_ratio:.1%})")
print(f"Target distribution: 95% negative, 5% positive")
print(f"Weight for class 0 (negative): {weight_neg:.3f}")
print(f"Weight for class 1 (positive): {weight_pos:.3f}")
print(f"Sample weights shape: {sample_weights.shape}")
print(f"Sample weights ready")

Train distribution: 50 negative (98.0%), 1 positive (2.0%)
Target distribution: 95% negative, 5% positive
Weight for class 0 (negative): 0.969
Weight for class 1 (positive): 2.550
Sample weights shape: (51,)
Sample weights ready


In [ ]:
models_sampling = {
    "xgb_weighted": {
        "samplers": [],
        "classifier": XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        ),
    },
    
    "rf_weighted": {
        "samplers": [],
        "classifier": RandomForestClassifier(
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        ),
    },
}

classifier_constructors = {
    "xgb": lambda: XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="aucpr",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
    
    "rf": lambda: RandomForestClassifier(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
}

param_spaces_clf = {
    "xgb": {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 4, 8),
        "learning_rate": ("float", 0.01, 0.1),
        "subsample": ("float", 0.7, 1.0),
        "colsample_bytree": ("float", 0.7, 1.0),
        "reg_alpha": ("float", 0.1, 1.0),
        "reg_lambda": ("float", 0.5, 5.0),
    },
    
    "rf": {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 5, 15),
        "min_samples_split": ("int", 5, 20),
        "min_samples_leaf": ("int", 2, 10),
    },
}

print("Classification models configured")

Classification models configured


In [ ]:
def create_classification_objective(name, samplers, X_train, y_train, cv):
    """Create Optuna objective - fixed for sklearn 1.8+"""
    
    model_type = get_model_type(name)
    
    def objective(trial):
        params = {}
        for param_name, param_config in param_spaces_clf[model_type].items():
            param_type = param_config[0]
            
            if param_type == "int":
                params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
            elif param_type == "float":
                params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2])
            elif param_type == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_config[1])
        
        clf = classifier_constructors[model_type]()
        clf.set_params(**params)
        
        scores = cross_val_score(
            clf, X_train, y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=N_JOBS
        )
        
        return scores.mean()
    
    return objective

print("Fixed objective function ready")

Fixed objective function ready


In [ ]:
cv_clf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_sampling = {}

for name, config in models_sampling.items():
    print(f"\n{'='*60}")
    print(f"Training: {name.upper()}")
    print(f"{'='*60}")
    
    model_type = get_model_type(name)
    
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{name}_classification"
    )
    
    objective = create_classification_objective(name, [], X_train_clf, y_train_clf, cv_clf)
    
    print(f"Running {N_TRIALS} trials...")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    if len(study.trials) == 0 or study.best_trial is None:
        print(f"WARNING: No trials completed for {name}, skipping...")
        continue
    
    best_params = study.best_params
    
    clf = classifier_constructors[model_type]()
    clf.set_params(**best_params)
    clf.fit(X_train_clf, y_train_clf, sample_weight=sample_weights)
    
    y_pred_proba = clf.predict_proba(X_test_clf)[:, 1]
    
    precisions, recalls, thresholds = precision_recall_curve(y_test_clf, y_pred_proba)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    best_threshold_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_threshold_idx] if len(thresholds) > best_threshold_idx else 0.5
    
    y_pred = (y_pred_proba >= best_threshold).astype(int)
    
    results_sampling[name] = {
        "study": study,
        "model": clf,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "cv_roc_auc": study.best_value,
        "test_recall": recall_score(y_test_clf, y_pred),
        "test_precision": precision_score(y_test_clf, y_pred, zero_division=0),
        "test_f1": f1_score(y_test_clf, y_pred),
        "test_pr_auc": average_precision_score(y_test_clf, y_pred_proba),
    }
    
    print(f"\nOptimization complete!")
    print(f"  Best CV ROC-AUC: {study.best_value:.4f}")
    print(f"  Test PR-AUC: {results_sampling[name]['test_pr_auc']:.4f}")
    print(f"  Test F1: {results_sampling[name]['test_f1']:.4f}")

[I 2026-01-27 21:13:02,373] A new study created in memory with name: xgb_weighted_classification



Training: XGB_WEIGHTED
Running 10 trials...


  0%|          | 0/10 [00:00<?, ?it/s]

[W 2026-01-27 21:13:02,642] Trial 0 failed with parameters: {'n_estimators': 187, 'max_depth': 6, 'learning_rate': 0.09789719303442712, 'subsample': 0.7266697064944387, 'colsample_bytree': 0.8437869579040544, 'reg_alpha': 0.988795444813903, 'reg_lambda': 1.3916349005851316} because of the following error: The value nan is not acceptable.
[W 2026-01-27 21:13:02,642] Trial 0 failed with value np.float64(nan).
[W 2026-01-27 21:13:02,903] Trial 1 failed with parameters: {'n_estimators': 465, 'max_depth': 6, 'learning_rate': 0.0804906521628185, 'subsample': 0.7907429768858647, 'colsample_bytree': 0.8191902980458703, 'reg_alpha': 0.36637058322138066, 'reg_lambda': 4.523398074565249} because of the following error: The value nan is not acceptable.
[W 2026-01-27 21:13:02,904] Trial 1 failed with value np.float64(nan).
[W 2026-01-27 21:13:03,116] Trial 2 failed with parameters: {'n_estimators': 119, 'max_depth': 6, 'learning_rate': 0.060570450028216496, 'subsample': 0.8091143890796986, 'colsamp

ValueError: No trials are completed yet.

In [ ]:
print(f"\n{'='*80}")
print(f"{'='*20} CLASSIFICATION SUMMARY {'='*20}")
print(f"{'='*80}\n")

print(f"{'Model':<20} {'CV PR-AUC':<12} {'Test PR-AUC':<12} {'Recall':<10} {'Precision':<12} {'F1':<10} {'Threshold':<10}")
print("-" * 110)

for name, result in results_sampling.items():
    cv_pr = result['cv_pr_auc']
    test_pr = result['test_pr_auc']
    recall = result['test_recall']
    precision = result['test_precision']
    f1 = result['test_f1']
    threshold = result['best_threshold']
    
    print(f"{name:<20} {cv_pr:<12.4f} {test_pr:<12.4f} {recall:<10.4f} {precision:<12.4f} {f1:<10.4f} {threshold:<10.3f}")

best_pr_auc_name = max(results_sampling.items(), key=lambda x: x[1]['test_pr_auc'])[0]
print(f"\n Best model by PR-AUC: {best_pr_auc_name.upper()} (PR-AUC: {results_sampling[best_pr_auc_name]['test_pr_auc']:.4f})")

best_f1_name = max(results_sampling.items(), key=lambda x: x[1]['test_f1'])[0]
print(f" Best model by F1: {best_f1_name.upper()} (F1: {results_sampling[best_f1_name]['test_f1']:.4f})")

## Save Best Models

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save best regression model
best_reg_name = min(results_regression.items(), key=lambda x: x[1]['test_rmse'])[0]
best_reg_model = results_regression[best_reg_name]["model"]

reg_filename = f"best_regressor_{best_reg_name}_{timestamp}.pkl"
joblib.dump(best_reg_model, reg_filename)
print(f"✅ Best regression model saved: {reg_filename}")

# Save best classification model
best_clf_name = max(results_sampling.items(), key=lambda x: x[1]['test_pr_auc'])[0]
best_clf_pipe = results_sampling[best_clf_name]["best_pipe"]

clf_filename = f"best_classifier_{best_clf_name}_{timestamp}.pkl"
joblib.dump(best_clf_pipe, clf_filename)
print(f"✅ Best classifier saved: {clf_filename}")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE!")
print(f"{'='*60}")